# Integrated Neuro-Symbolic Pipeline: Baseline CNN vs CBM + GFlowNet + Rule Regularization

Notebook mỏng (launcher) chạy liên hoàn 6 giai đoạn có **Baseline đối chứng**:
1. **Sanity Check:** Thử nghiệm 1 iteration (`next(iter(dataset))`) để đảm bảo pipeline thông suốt.
2. **Stage 1:** Huấn luyện **Baseline CNN** (MobileNetV3) làm chuẩn đối chứng (Black-box).
3. **Stage 1b:** Huấn luyện **CBM** ép vector đặc trưng về 61 concept có xác suất $[0, 1]$.
4. **Stage 2:** Trích xuất ma trận xác suất concept $P(c) \in [0, 1]$ từ CBM.
5. **Stage 3:** Trích xuất tập luật ngữ nghĩa bằng Random Forest ($P(\text{concept}) \to \text{Class}$).
6. **Stage 4:** GFlowNet tuyển chọn tập luật Concept tối ưu, đa dạng, không loãng.
7. **Stage 5:** Huấn luyện CBM có ràng buộc luật (Rule-Regularized Fine-Tuning).

In [ ]:
# =========================================================================
# THAM SỐ THỰC NGHIỆM TỰ ĐỘNG SINH BỞI GENERATOR SCRIPT
# =========================================================================
DATA_DIR = '/kaggle/input/datasets/dangduykhoab2207532/mykhmer/khmer'
CONCEPT_ANNOTATIONS_PATH = '/kaggle/input/datasets/dangduykhoab2207532/mykhmer/image_concepts.json'
NUM_CLASSES = 9
N_CONCEPTS = 61
SEED = 42
BACKBONE = 'mobilenetv3_small'
CONCEPT_WEIGHT = 1.0
NUM_EPOCHS = 50
PATIENCE = 5
BATCH_SIZE = 64
GIT_REPO = 'https://github.com/khoaddb2207532/neuro_symbolic_mlops_l2_app.git'
GIT_BRANCH = 'main'


In [ ]:
# Kiểm tra dữ liệu ảnh và file nhãn concept đầu vào
from pathlib import Path

data_path = Path(DATA_DIR)
concept_path = Path(CONCEPT_ANNOTATIONS_PATH)

if not data_path.exists():
    possible_data = list(Path("/kaggle/input").glob("**/khmer"))
    if possible_data:
        DATA_DIR = str(possible_data[0])
        print(f"Tự động phát hiện DATA_DIR tại: {DATA_DIR}")
    else:
        print(f"CẢNH BÁO: Không tìm thấy DATA_DIR: {DATA_DIR}")
else:
    print(f"DATA_DIR hợp lệ tại: {DATA_DIR}")

if not concept_path.exists():
    possible_json = list(Path("/kaggle/input").glob("**/*concept*.json"))
    if possible_json:
        CONCEPT_ANNOTATIONS_PATH = str(possible_json[0])
        print(f"Tự động phát hiện file concept tại: {CONCEPT_ANNOTATIONS_PATH}")
    else:
        print(f"CẢNH BÁO: Không tìm thấy file concept: {CONCEPT_ANNOTATIONS_PATH}")
else:
    print(f"File concept hợp lệ tại: {CONCEPT_ANNOTATIONS_PATH}")

In [ ]:
# Clone repo & setup môi trường
import subprocess
import sys
from pathlib import Path

PROJECT = Path("/kaggle/working/neuro_symbolic_mlops_l2_app")
if not PROJECT.exists():
    print("Đang clone repo mới nhất...")
    subprocess.run(["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_REPO, str(PROJECT)], check=True)
print("Repo sẵn sàng tại:", PROJECT)

In [ ]:
# Cài đặt thư viện phụ thuộc
%cd /kaggle/working/neuro_symbolic_mlops_l2_app
!pip install -q torchgfn tensordict openpyxl

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))

In [ ]:
# Cập nhật file params.yaml từ tham số notebook
import yaml

with open("params.yaml", "r", encoding="utf-8") as f:
    params = yaml.safe_load(f)

params["data_dir"] = DATA_DIR
params["num_classes"] = NUM_CLASSES
params["seed"] = SEED
params["num_epochs"] = NUM_EPOCHS
params["patience"] = PATIENCE
params["batch_size"] = BATCH_SIZE
params["baseline_comparison"]["selected_architecture"] = BACKBONE

# Thiết lập CBM với 61 concept và file nhãn VLM
params.setdefault("cbm", {})
params["cbm"]["n_concepts"] = N_CONCEPTS
params["cbm"]["concept_weight"] = CONCEPT_WEIGHT
params["cbm"]["concept_annotations_path"] = CONCEPT_ANNOTATIONS_PATH
params.setdefault("features", {})
params["features"]["source"] = "cbm"
params.setdefault("rule_penalty", {})
params["rule_penalty"]["target_model"] = "cbm"

with open("params.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(params, f, sort_keys=False, allow_unicode=True)

print(f"params.yaml đã cấu hình: {NUM_CLASSES} classes, {N_CONCEPTS} concepts, source=cbm, target_model=cbm!")

In [ ]:
# =========================================================================
# SANITY CHECK / SMOKE TEST: Chạy thử 1 iteration (1 batch) trước khi chạy thật
# =========================================================================
import torch
from src.data.dataset import create_dataloaders
from src.data.concepts import ConceptAnnotatedDataset, load_concept_annotations
from src.models.cnn import ImageClassificationBaseline
from src.models.cbm import ConceptBottleneckModel

print(">>> ĐANG CHẠY SANITY CHECK TRÊN 1 ITERATION CỦA DATASET & MODELS...")
dataloaders, train_loader, val_loader, test_loader = create_dataloaders(
    DATA_DIR, batch_size=BATCH_SIZE, num_workers=2, seed=SEED
)

# 1. Test 1 iteration qua DataLoader ảnh gốc
images, labels = next(iter(train_loader))
print(f"  [1/4] Batch ảnh train: images={tuple(images.shape)}, labels={tuple(labels.shape)}")
assert images.shape[0] == BATCH_SIZE, 'Lỗi kích thước batch!'

# 2. Test nạp concept annotations
ann, names, n_c, class_fb = load_concept_annotations(CONCEPT_ANNOTATIONS_PATH)
c_ds = ConceptAnnotatedDataset(train_loader.dataset, ann, class_fb, n_concepts=n_c)
c_img, c_lbl, c_vec = c_ds[0]
print(f"  [2/4] Mẫu ConceptDataset: image={tuple(c_img.shape)}, label={c_lbl}, concept_vector={tuple(c_vec.shape)}")
print(f"        Số concept được kích hoạt ở mẫu đầu tiên: {int(c_vec.sum().item())}/{n_c}")

# 3. Test forward pass qua Baseline CNN
device = 'cuda' if torch.cuda.is_available() else 'cpu'
baseline = ImageClassificationBaseline(architecture=BACKBONE, num_classes=NUM_CLASSES, pretrained=False).to(device)
with torch.no_grad():
    b_logits, b_feats = baseline(images.to(device))
print(f"  [3/4] Baseline CNN forward: logits={tuple(b_logits.shape)}, features={tuple(b_feats.shape)}")

# 4. Test forward pass qua CBM (ép về [0, 1])
cbm = ConceptBottleneckModel(architecture=BACKBONE, n_concepts=n_c, num_classes=NUM_CLASSES, pretrained=False, concept_names=names).to(device)
with torch.no_grad():
    c_logits, c_probs, c_feats = cbm(images.to(device))
print(f"  [4/4] CBM forward: logits={tuple(c_logits.shape)}, concepts={tuple(c_probs.shape)} (min={c_probs.min():.2f}, max={c_probs.max():.2f})")

print("\n>>> SANITY CHECK THÀNH CÔNG 100%! HỆ THỐNG ĐÃ SẴN SÀNG HUẤN LUYỆN CHÍNH THỨC.")

In [ ]:
# Chạy toàn bộ pipeline tích hợp 6 giai đoạn (Có Baseline đối chứng)
!python -m pipelines.run_cbm_neuro_symbolic_pipeline --config params.yaml

In [ ]:
# So sánh hiệu năng đối chứng giữa Baseline CNN vs CBM vs CBM + Rule
import json
from pathlib import Path
import pandas as pd

print("=== BẢNG SO SÁNH HIỆU NĂNG ĐỐI CHỨNG (ABLATION STUDY) ===")
results = []

# 1. Baseline
b_report = Path("outputs/01_baseline/baseline_cnn_performance_classification_report.txt")
if b_report.exists():
    line = b_report.read_text(encoding='utf-8').splitlines()[0]
    results.append({"Model": "1. Baseline CNN (MobileNetV3)", "Type": "Hộp đen (1280-d)", "Result": line})

# 2. CBM
c_report = Path("outputs/cbm/cbm_performance_classification_report.txt")
if c_report.exists():
    line = c_report.read_text(encoding='utf-8').splitlines()[0]
    results.append({"Model": "2. CBM (Concept Bottleneck)", "Type": "Bán giải thích (61 concept)", "Result": line})

# 3. Rule-Regularized CBM
r_report = Path("outputs/05_rules_model/rule-regularized_cnn_performance_classification_report.txt")
if r_report.exists():
    line = r_report.read_text(encoding='utf-8').splitlines()[0]
    results.append({"Model": "3. CBM + GFlowNet Rule-Regularized", "Type": "Neuro-Symbolic khép kín", "Result": line})

if results:
    display(pd.DataFrame(results))
else:
    print("Chưa hoàn tất huấn luyện để tổng hợp bảng.")

In [ ]:
# Hiển thị Top luật Concept được GFlowNet tuyển chọn (Người đọc hiểu 100%)
import pandas as pd
from pathlib import Path

rules_excel = Path("outputs/04_filtered_rules/selected_rules.xlsx")
if rules_excel.exists():
    df = pd.read_excel(rules_excel)
    print(f"\n=== TỔNG CỘNG CÓ {len(df)} LUẬT CONCEPT ĐƯỢC GFLOWNET CHỌN ===")
    display(df.head(15))
else:
    print("Chưa tìm thấy selected_rules.xlsx")

In [ ]:
# Đóng gói kết quả đầu ra thành file zip để tải về
!zip -r -q /kaggle/working/cbm_neuro_symbolic_outputs.zip outputs/
print("\nĐÃ NÉN TOÀN BỘ KẾT QUẢ TẠI: /kaggle/working/cbm_neuro_symbolic_outputs.zip")
!ls -lh /kaggle/working/cbm_neuro_symbolic_outputs.zip